# 1. Preparação (dados, tokenizer, métricas)

Carrega o dataset AGNews (fold 0) a partir do split de Claudio Valiense, separa validação e teste, e prepara o tokenizer BERT e as métricas. Esta seção é a base comum para os três modelos.

In [ ]:
!pip install jsonlines
!pip install numpy
!pip install evaluate

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import evaluate

In [ ]:
dataset_name = "agnews"

index_fold = 0

batch_size = 8

max_length = 64

num_labels = 4

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
base_path = "/kaggle/input/datasets/claudiovaliense/datasets-lbd3/agnews"

with open(
    f"{base_path}/texts.txt",
    "r",
    encoding="utf-8"
) as f:
    texts = [line.strip() for line in f]

with open(
    f"{base_path}/score.txt",
    "r",
    encoding="utf-8"
) as f:
    labels = [int(line.strip()) for line in f]

df_full = pd.DataFrame({
    "text": texts,
    "label": labels
})

print(df_full.shape)
display(df_full.head())

In [ ]:
split_df = pd.read_pickle(
    "/kaggle/input/datasets/claudiovaliense/datasets-lbd3/agnews/splits/split_5_with_val.pkl"
)

fold = split_df.iloc[index_fold]

print("train:", len(fold["train_idxs"]))
print("val:", len(fold["val_idxs"]))
print("test:", len(fold["test_idxs"]))

In [ ]:
val_df = df_full.iloc[
    fold["val_idxs"]
].reset_index(drop=True)

test_df = df_full.iloc[
    fold["test_idxs"]
].reset_index(drop=True)

val_df["label"] = val_df["label"] - 1
test_df["label"] = test_df["label"] - 1

print(val_df.shape)
print(test_df.shape)

In [ ]:
print(sorted(val_df["label"].unique()))
print(sorted(test_df["label"].unique()))

## 2. Treino E2SC

Treina o BERT no conjunto reduzido pelo método E2SC (foco em redundância). Carrega o parquet do E2SC, tokeniza (max_length=64), treina por 2 épocas e salva o modelo como `agnews_e2sc_model`.

In [ ]:
import glob
p = glob.glob("/kaggle/input/**/e2sc_a25/agnews/train_fold_0.parquet", recursive=True)[0]
df_train = pd.read_parquet(p)

# normaliza para 0-3 só se vier em 1-4
if df_train["label"].min() == 1:
    df_train["label"] = df_train["label"] - 1

print(df_train.shape)
print(sorted(df_train["label"].unique()))
df_train["label"].value_counts()

In [ ]:
train_dataset = Dataset.from_pandas(df_train)

val_dataset = Dataset.from_pandas(val_df)

test_dataset = Dataset.from_pandas(test_df)

print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

In [ ]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
accuracy_metric = evaluate.load("accuracy")

f1_metric = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"
    )

    return {
        "accuracy": accuracy["accuracy"],
        "f1": f1["f1"]
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",                      # <-- não salva checkpoint por época (economiza tempo/disco)
    learning_rate=2e-5,
    per_device_train_batch_size=32,          # <-- maior batch = mais rápido na GPU (era batch_size variável)
    per_device_eval_batch_size=64,
    num_train_epochs=2,                      # 2 épocas já basta pro AGNews
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),          # já estava — usa meia precisão, mais rápido
    report_to="none",                        # <-- não tenta logar em wandb etc
)

In [ ]:
import transformers

print(transformers.__version__)

In [ ]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_train,

    eval_dataset=tokenized_val,

    compute_metrics=compute_metrics
)

In [ ]:
print(df_train["label"].unique())
print(sorted(df_train["label"].unique()))
print(df_train["label"].value_counts())

In [ ]:
print(sorted(val_df["label"].unique()))
print(sorted(test_df["label"].unique()))

print(
    val_df["label"].min(),
    val_df["label"].max()
)

print(
    test_df["label"].min(),
    test_df["label"].max()
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate()

print(results)

In [ ]:
trainer.save_model("/kaggle/working/agnews_e2sc_model")
tokenizer.save_pretrained("/kaggle/working/agnews_e2sc_model")
print("E2SC salvo!")

## 3. Treino biO-IS

Treina o BERT no conjunto reduzido pelo método biO-IS (foco em ruído e redundância), com um modelo novo e independente do E2SC. Salva como `agnews_biois_model`.

In [ ]:
import glob
p_biois = glob.glob("/kaggle/input/**/biois_a25_t50/agnews/train_fold_0.parquet", recursive=True)[0]
df_train_biois = pd.read_parquet(p_biois)

# normaliza para 0-3 só se vier em 1-4 (mesmo cuidado do E2SC)
if df_train_biois["label"].min() == 1:
    df_train_biois["label"] = df_train_biois["label"] - 1

print(df_train_biois.shape)
print(sorted(df_train_biois["label"].unique()))   # tem que dar [0, 1, 2, 3]
df_train_biois["label"].value_counts()

In [ ]:
train_dataset_biois = Dataset.from_pandas(df_train_biois)

tokenized_train_biois = train_dataset_biois.map(
    tokenize_function,
    batched=True
)
print(train_dataset_biois)

In [ ]:
model_biois = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
trainer_biois = Trainer(
    model=model_biois,
    args=training_args,                    # reaproveita os mesmos args da seção 1
    train_dataset=tokenized_train_biois,
    eval_dataset=tokenized_val,            # mesma validação do E2SC
    compute_metrics=compute_metrics
)

In [ ]:
trainer_biois.train()

In [ ]:
results_biois = trainer_biois.evaluate()
print(results_biois)

trainer_biois.save_model("/kaggle/working/agnews_biois_model")
tokenizer.save_pretrained("/kaggle/working/agnews_biois_model")
print("biO-IS salvo!")

## 4. Extração de embeddings

Extrai os embeddings da última camada do BERT (vetor [CLS]) para o conjunto de teste, em cada modelo (E2SC e biO-IS), no mesmo formato do baseline. Salva os `.pkl` de representação e verifica o alinhamento dos três cenários.

In [ ]:
import torch
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# conjunto de teste no formato do baseline (coluna "labels")
x_test = test_df.copy().rename(columns={"label": "labels"})

# tokeniza o teste uma vez (reaproveitado pelos três modelos)
test_encodings = tokenizer(
    list(x_test["text"]),
    max_length=256,
    return_tensors="pt",
    padding=True,
    truncation=True
)

class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_dataset_emb = CustomDataset(test_encodings, list(x_test["labels"]))
test_dataloader = DataLoader(test_dataset_emb, batch_size=32)

def extrair_embeddings(model):
    model.eval()
    model.to(device)
    embeddings, preds = [], []
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
            cls = outputs.last_hidden_state[:, 0, :]
            embeddings.extend(cls.cpu().numpy())
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
    df = x_test.copy()
    df["bert"] = embeddings
    df["pred"] = preds
    return df

In [ ]:
#extração do e2sc
df_e2sc_repr = extrair_embeddings(trainer.model)
df_e2sc_repr.to_pickle("/kaggle/working/agnews_e2sc_representation.pkl")
print("E2SC embeddings salvos:", df_e2sc_repr.shape)
print(df_e2sc_repr[["labels", "pred"]].head())

In [ ]:
#extração do bios
df_biois_repr = extrair_embeddings(trainer_biois.model)
df_biois_repr.to_pickle("/kaggle/working/agnews_biois_representation.pkl")
print("biO-IS embeddings salvos:", df_biois_repr.shape)
print(df_biois_repr[["labels", "pred"]].head())

In [ ]:
import pandas as pd

base = pd.read_pickle("/kaggle/input/notebooks/karinabatista/2bert-liziane-modelo-atencao-abr2025/agnews_f0_bert_representation.pkl")
e2sc = pd.read_pickle("/kaggle/working/agnews_e2sc_representation.pkl")
biois = pd.read_pickle("/kaggle/working/agnews_biois_representation.pkl")

for nome, df in [("baseline", base), ("E2SC", e2sc), ("biO-IS", biois)]:
    acc = (df["pred"] == df["labels"]).mean()
    print(f"{nome:9s} | shape {df.shape} | acurácia teste: {acc:.4f}")

# confirma que os três estão na mesma ordem (mesmos textos)
print("\nmesma ordem (baseline vs E2SC):", (base["text"].values == e2sc["text"].values).all())
print("mesma ordem (baseline vs biO-IS):", (base["text"].values == biois["text"].values).all())